# Multi-Branch Keras — Modality-Specific Sequence Classifier

This notebook uses a multi-branch deep learning architecture where each sensor modality (Acceleration, Rotation, TOF, Thermopile, STFT, CWT) has its own dedicated branch. 

It follows the exact style of the Random Forest notebook:
- configuration
- data loading
- split
- estimator (Keras Multi-Branch)
- parameter search (grid or Bayesian)
- holdout evaluation
- save results

In [1]:
import os
import sys
import warnings
import json
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Local path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Kaggle path routing
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception:
    pass

from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

import data_utils
import multibranch_attention as multibranch_keras
importlib.reload(multibranch_keras)
from base_utils_qwen import (
    SequenceExtractor,
    SensorAugmentor,
    evaluate_holdout,
    prepare_bayesian_space
)
print('Imports loaded successfully')

Imports loaded successfully


In [2]:
experiment_name = 'multibranch_attention'

# Supported targets: 'bfrb', 'gesture_action', 'orientation', 'gesture'
TARGET_COL = 'bfrb'

# Use sample.csv for a quick smoke test; set False for full train.csv
use_sample_data = False
sample_file = 'sample.csv'

search_mode = 'bayesian'  # 'grid' or 'bayesian'
random_state = 42
n_splits = 1              # 1 -> GroupShuffleSplit; >=2 -> GroupKFold
cv_test_size = 0.4
train_size = 0.6
n_iter = 15
verbose = 4
error_score = 'raise'
n_jobs = -1                 # TensorFlow models should not be fitted in parallel

# Leave empty to keep all orientations, especially for orientation prediction.
orientation_filter_list = []
problematic_sequence_bool = True

target_only_bool = False

epochs = 15
batch_size = 64

results_dir = Path(workspace_root) / f'results_{experiment_name}_{TARGET_COL}_{search_mode}'
results_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=cv_test_size,
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)

In [3]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

train_demo_path = data_root / 'train_demographics.csv'
train_demo_df = pd.read_csv(train_demo_path) if train_demo_path.exists() else None

# Derive all supported sequence targets using the same definitions as random_forest.ipynb.
train_df = raw_train_df.copy()
train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')
train_df['gesture_position'] = train_df['gesture'].str.split(' - ').str[0]
train_df['gesture_action'] = train_df['gesture'].str.split(' - ').str[-1]

if TARGET_COL not in {'bfrb', 'gesture_action', 'orientation', 'gesture'}:
    raise ValueError("TARGET_COL must be one of 'bfrb', 'gesture_action', 'orientation', or 'gesture'.")

if orientation_filter_list:
    train_df = train_df[train_df['orientation'].isin(orientation_filter_list)].copy()
if target_only_bool:
    train_df = train_df[train_df['is_target'].astype(bool)].copy()

train_df[TARGET_COL] = train_df[TARGET_COL].fillna('non_bfrb').astype(str)
print(f'Target: {TARGET_COL}; sequences after filters: {train_df["sequence_id"].nunique()}')

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data
Using train.csv: 8151 sequences
Target: bfrb; sequences after filters: 8151


In [4]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=min(0.2, 1 - train_size),
        random_state=random_state,
    )
except Exception as split_error:
    print(f'Balanced split unavailable ({split_error}); using grouped random split.')
    sequence_df = (
        train_df[['sequence_id', 'is_target', TARGET_COL]]
        .drop_duplicates('sequence_id')
        .sort_values('sequence_id')
    )
    splitter = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(sequence_df, groups=sequence_df['sequence_id']))
    train_ids = sequence_df.iloc[train_idx]['sequence_id']
    test_ids = sequence_df.iloc[test_idx]['sequence_id']
    train_sample_df = train_df[train_df['sequence_id'].isin(train_ids)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_ids)].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = X_train[['sequence_id', 'is_target', TARGET_COL]].copy()
y_test = X_test[['sequence_id', 'is_target', TARGET_COL]].copy()
groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Holdout sequences:', X_test['sequence_id'].nunique())

Train: 3879 seqs | 47.6%
Test:  966 seqs  | 11.9%
Train sequences: 3879
Holdout sequences: 966


In [5]:
extractor = SequenceExtractor(
    output_format='chunks',
    acc_modes='smoothed|velocity|displacement|jerk',
    rotation_modes='quaternion|euler|angular_velocity',
    tof_modes='pooled_stats|sensor_stats',
    thm_modes='centered_diff',
    frame_stats='mean,std,min,max,last,first,rms',
)

clf = multibranch_keras.MultiBranchSequenceClassifier(
    extractor=extractor,
    primary_target=TARGET_COL,
    epochs=epochs,
    batch_size=batch_size,
    random_state=random_state,
)

pipeline = Pipeline([
    ('augmentor', SensorAugmentor(
        sequence_col='sequence_id',
        counter_col='sequence_counter',
        prob=0.0,
        per_aug_prob=0.0,
        seed=random_state,
    )),
    ('estimator', clf),
])

In [6]:
SEARCH_BRANCH_FILTERS = [
    {'acc': 16, 'rotation': 16, 'tof': 16, 'thermo': 16, 'misc': 16},
    {'acc': 32, 'rotation': 32, 'tof': 32, 'thermo': 32, 'misc': 32},
    {'acc': 64, 'rotation': 64, 'tof': 64, 'thermo': 32, 'misc': 64},
    {'acc': 128, 'rotation': 128, 'tof': 128, 'thermo': 32, 'misc': 128},
    {'acc': 256, 'rotation': 256, 'tof': 256, 'thermo': 32, 'misc': 256},

]

GRID_PARAM_SPACE = {
    'augmentor__prob': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__per_aug_prob': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__sensor_drop_prob': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__noise_std': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__jitter_sigma': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__scaling_sigma': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__channel_drop_prob': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__time_shift_frac': [0.0, 0.1, 0.2],
    'augmentor__temporal_num_masks': [0, 1, 2],
    'augmentor__temporal_mask_frac': [0.0, 0.1, 0.2],
    'estimator__extractor__acc_modes': ['smoothed|velocity|displacement|jerk'],
    'estimator__extractor__rotation_modes': ['quaternion|euler|angular_velocity'],
    'estimator__extractor__tof_modes': ['pooled_stats|sensor_stats'],
    'estimator__extractor__thm_modes': ['centered_diff'],
    'estimator__extractor__frame_stats': ['mean,std,min,max,last,first,rms'],
    'estimator__extractor__motion_filter_mode': [None],
    'estimator__extractor__use_dead_reckoning': [False],
    'estimator__extractor__dead_reckoning_detrend': [False],
    'estimator__extractor__kalman_process_noise': [1e-3],
    'estimator__extractor__kalman_measurement_noise': [1e-1],
    'estimator__extractor__window_size': [5],
    'estimator__extractor__smooth_alpha': [None],
    'estimator__extractor__clip_value': [None],
    'estimator__extractor__interp_mode': ['linear'],
    'estimator__extractor__output_format': ['chunks'],
    'estimator__extractor__padding_value': [0.0],
    'estimator__extractor__maxlen': [160],
    'estimator__extractor__chunk_window_size': [100],
    'estimator__extractor__chunk_stride': [50],
    'estimator__extractor__add_global_context': [False, True],
    'estimator__extractor__resample_modalities': [False, True],
    'estimator__extractor__compute_dt': [True],
    'estimator__extractor__imu_native_sampling_rate': [100],
    'estimator__extractor__rot_native_sampling_rate': [100],
    'estimator__extractor__tof_native_sampling_rate': [20],
    'estimator__extractor__thm_native_sampling_rate': [20],
    'estimator__extractor__imu_target_sampling_rate': [100],
    'estimator__extractor__rot_target_sampling_rate': [100],
    'estimator__extractor__tof_target_sampling_rate': [20],
    'estimator__extractor__thm_target_sampling_rate': [20],
    'estimator__extractor__stft_use_log_scale': [True],
    'estimator__extractor__cwt_use_log_scale': [True],
    'estimator__extractor__filter_problematic_sequences': [problematic_sequence_bool],
    'estimator__extractor__ideal_skew_threshold': [1.0, 1.1, 1.2, 1.3, 1.4, 1.5],
    'estimator__extractor__problematic_features_threshold': [4, 5, 6, 7],
    'estimator__branch_filters': SEARCH_BRANCH_FILTERS,
    'estimator__dense_units': [32, 64, 128],
    'estimator__dropout': [0.2, 0.4],
    'estimator__learning_rate': [1e-3, 1e-4],
    'estimator__epochs': [10, 30],
    'estimator__batch_size': [32, 64, 128],
}

if SKOPT_AVAILABLE:
    BAYESIAN_PARAM_SPACE = multibranch_keras.prepare_multibranch_bayesian_space(
        GRID_PARAM_SPACE
    )
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

for space_name, candidate_space in (
    ('grid', GRID_PARAM_SPACE),
    ('bayesian', BAYESIAN_PARAM_SPACE),
):
    unknown_params = set(candidate_space) - set(pipeline.get_params(deep=True))
    if unknown_params:
        raise ValueError(
            f'Unknown {space_name} search parameters: {sorted(unknown_params)}'
        )

param_space = BAYESIAN_PARAM_SPACE if search_mode == 'bayesian' else GRID_PARAM_SPACE
print(f'Validated {len(GRID_PARAM_SPACE)} Grid and Bayesian search parameters.')

Validated 51 Grid and Bayesian search parameters.


In [7]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )
    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=None,  # Pipeline delegates to sequence-level estimator.score().
        cv=cv_object,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=None,
        cv=cv_object,
        n_jobs=n_jobs,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )

search.fit(X_train, y_train, groups=groups)

print('Best CV score:', search.best_score_)
print('Best params:', search.best_params_)

Fitting 1 folds for each of 1 candidates, totalling 1 fits


I0000 00:00:1790558958.566125      65 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790558958.568947      65 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1790558979.108174      86 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Fitting 1 folds for each of 1 candidates, totalling 1 fits
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END augmentor__channel_drop_prob=0.3, augmentor__jitter_sigma=0.5, augmentor__noise_std=0.6, augmentor__per_aug_prob=0.1, augmentor__prob=0.5, augmentor__scaling_sigma=0.3, augmentor__sensor_drop_prob=0.1, augmentor__temporal_mask_frac=0.2, augmentor__temporal_num_masks=0, augmentor__time_shift_frac=0.1, estimator__batch_size=64, estimator__branch_filters={"acc": 16, "misc": 16, "rotation": 16, "thermo": 16, "tof": 16}, estimator__dense_units=32, estimator__dropout=0.4, estimator__epochs=10, estimator__extractor__acc_modes=smoothed|velocity|displacement|jerk, estimator__extractor__add_global_context=True, estimator__extractor__chunk_stride=50, estimator__extractor__chunk_window_size=100, estimator__extractor__clip_value=None, estimator__extractor__compute_dt=True, estimator__extractor__cwt_use_log_scale=True, estimator__extractor__dead_reckoning_detrend=False, 

I0000 00:00:1790561048.842640    1226 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790561048.845022    1226 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1790561068.202459    1245 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Fitting 1 folds for each of 1 candidates, totalling 1 fits
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END augmentor__channel_drop_prob=0.3, augmentor__jitter_sigma=0.6, augmentor__noise_std=0.0, augmentor__per_aug_prob=0.3, augmentor__prob=0.0, augmentor__scaling_sigma=0.3, augmentor__sensor_drop_prob=0.0, augmentor__temporal_mask_frac=0.2, augmentor__temporal_num_masks=1, augmentor__time_shift_frac=0.2, estimator__batch_size=128, estimator__branch_filters={"acc": 32, "misc": 32, "rotation": 32, "thermo": 32, "tof": 32}, estimator__dense_units=32, estimator__dropout=0.4, estimator__epochs=10, estimator__extractor__acc_modes=smoothed|velocity|displacement|jerk, estimator__extractor__add_global_context=True, estimator__extractor__chunk_stride=50, estimator__extractor__chunk_window_size=100, estimator__extractor__clip_value=None, estimator__extractor__compute_dt=True, estimator__extractor__cwt_use_log_scale=True, estimator__extractor__dead_reckoning_detrend=False,

...

Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END augmentor__channel_drop_prob=0.5, augmentor__jitter_sigma=0.6, augmentor__noise_std=0.3, augmentor__per_aug_prob=0.3, augmentor__prob=0.1, augmentor__scaling_sigma=0.0, augmentor__sensor_drop_prob=0.0, augmentor__temporal_mask_frac=0.0, augmentor__temporal_num_masks=1, augmentor__time_shift_frac=0.1, estimator__batch_size=128, estimator__branch_filters={"acc": 256, "misc": 256, "rotation": 256, "thermo": 32, "tof": 256}, estimator__dense_units=128, estimator__dropout=0.4, estimator__epochs=30, estimator__extractor__acc_modes=smoothed|velocity|displacement|jerk, estimator__extractor__add_global_context=False, estimator__extractor__chunk_stride=50, estimator__extractor__chunk_window_size=100, estimator__extractor__clip_value=None, estimator__extractor__compute_dt=True, estimator__extractor__cwt_use_log_scale=True, estimator__extractor__dead_reckoning_detrend=False, estimator__extractor__filter_problematic_sequences=T

I0000 00:00:1790574335.360911    6278 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790574335.363194    6278 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1790574354.996126    6905 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


[CV 1/1] END augmentor__channel_drop_prob=0.1, augmentor__jitter_sigma=0.0, augmentor__noise_std=0.5, augmentor__per_aug_prob=0.1, augmentor__prob=0.5, augmentor__scaling_sigma=0.1, augmentor__sensor_drop_prob=0.3, augmentor__temporal_mask_frac=0.2, augmentor__temporal_num_masks=1, augmentor__time_shift_frac=0.1, estimator__batch_size=64, estimator__branch_filters={"acc": 16, "misc": 16, "rotation": 16, "thermo": 16, "tof": 16}, estimator__dense_units=128, estimator__dropout=0.4, estimator__epochs=10, estimator__extractor__acc_modes=smoothed|velocity|displacement|jerk, estimator__extractor__add_global_context=True, estimator__extractor__chunk_stride=50, estimator__extractor__chunk_window_size=100, estimator__extractor__clip_value=None, estimator__extractor__compute_dt=True, estimator__extractor__cwt_use_log_scale=True, estimator__extractor__dead_reckoning_detrend=False, estimator__extractor__filter_problematic_sequences=True, estimator__extractor__frame_stats=mean,std,min,max,last,firs

I0000 00:00:1790576220.041210      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790576220.046547      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1790576240.522165    7267 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Best CV score: 0.6425027805775487
Best params: OrderedDict({'augmentor__channel_drop_prob': 0.5, 'augmentor__jitter_sigma': 0.6, 'augmentor__noise_std': 0.0, 'augmentor__per_aug_prob': 0.0, 'augmentor__prob': 0.6, 'augmentor__scaling_sigma': 0.1, 'augmentor__sensor_drop_prob': 0.3, 'augmentor__temporal_mask_frac': 0.1, 'augmentor__temporal_num_masks': 2, 'augmentor__time_shift_frac': 0.1, 'estimator__batch_size': 64, 'estimator__branch_filters': '{"acc": 64, "misc": 64, "rotation": 64, "thermo": 32, "tof": 64}', 'estimator__dense_units': 32, 'estimator__dropout': 0.2, 'estimator__epochs': 30, 'estimator__extractor__acc_modes': 'smoothed|velocity|displacement|jerk', 'estimator__extractor__add_global_context': True, 'estimator__extractor__chunk_stride': 50, 'estimator__extractor__chunk_window_size': 100, 'estimator__extractor__clip_value': None, 'estimator__extractor__compute_dt': True, 'estimator__extractor__cwt_use_log_scale': True, 'estimator__extractor__dead_reckoning_detrend': False

In [8]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score / macro F1:', eval_results['competition_score'])


FINAL EVALUATION
Binary F1 (non_bfrb vs bfrb): 0.9587
BFRB Gesture Macro F1: 0.3805
COMPETITION SCORE: 0.6696

----------------------------------------
BFRB Gesture Classification Report
----------------------------------------
                          precision    recall  f1-score   support

   Above ear - pull hair       0.59      0.59      0.59        81
      Cheek - pinch skin       0.58      0.09      0.15        81
     Eyebrow - pull hair       0.30      0.30      0.30        81
     Eyelash - pull hair       0.34      0.67      0.45        81
Forehead - pull hairline       0.65      0.48      0.55        81
      Forehead - scratch       0.56      0.63      0.59        81
       Neck - pinch skin       0.40      0.47      0.43        81
          Neck - scratch       0.58      0.26      0.36        81
                non_bfrb       0.00      0.00      0.00         0

                accuracy                           0.44       648
               macro avg       0.44      0.

In [9]:
save_dir = results_dir / timestamp
save_dir.mkdir(parents=True, exist_ok=True)
best_estimator = best_model.named_steps['estimator']
best_estimator.model_.save(save_dir / 'best_model.h5')

pd.DataFrame(search.cv_results_).to_csv(
    save_dir / f'{experiment_name}_cv.csv',
    index=False,
)
eval_results['results_df'].to_csv(
    save_dir / f'{experiment_name}_holdout.csv',
    index=False,
)
pd.DataFrame([{
    'target': TARGET_COL,
    'best_score': search.best_score_,
    'best_params': json.dumps(search.best_params_, default=str),
    'holdout_score': eval_results['competition_score'],
}]).to_csv(
    save_dir / f'{experiment_name}_best.csv',
    index=False,
)

with open(save_dir / 'results.json', 'w') as results_file:
    json.dump({
        'target': TARGET_COL,
        'best_score': float(search.best_score_),
        'best_params': search.best_params_,
        'holdout_score': float(eval_results['competition_score']),
        'random_state': random_state,
        'train_sequences': int(X_train['sequence_id'].nunique()),
        'holdout_sequences': int(X_test['sequence_id'].nunique()),
    }, results_file, indent=2, default=str)

print(f'Results saved to {save_dir}')

Results saved to /kaggle/working/results_multibranch_attention_bfrb_bayesian/20260928_011531
